# berts-gg-qa-submit

Offline inference notebook.

Expected Kaggle inputs:
- competition data: `/kaggle/input/google-quest-challenge`
- model checkpoints + artifacts exported from training notebook (uploaded as dataset)


In [ ]:
import os, gc, json, html
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel


In [ ]:
DATA_DIR = '/kaggle/input/google-quest-challenge'
MODEL_DATASET_DIR = '/kaggle/input/ggqa-trained-models'  # dataset containing *.pt files

test = pd.read_csv(f'{DATA_DIR}/test.csv')
train = pd.read_csv(f'{DATA_DIR}/train.csv')
sample = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

target_cols = [c for c in train.columns if c not in test.columns and c != 'qa_id']
label_values = {c: sorted(train[c].dropna().unique().tolist()) for c in target_cols}

# fallback clipping config (works even when no artifacts were exported)
clippings = {
    'question_has_commonly_accepted_answer': [0.0, 0.6],
    'question_conversational': [0.15, 1.0],
    'question_multi_intent': [0.1, 1.0],
    'question_type_choice': [0.1, 1.0],
    'question_type_compare': [0.1, 1.0],
    'question_type_consequence': [0.08, 1.0],
    'question_type_definition': [0.1, 1.0],
    'question_type_entity': [0.13, 1.0]
}

# optional artifacts loading (if present in uploaded model dataset)
art_dir = f'{MODEL_DATASET_DIR}/artifacts'
if os.path.exists(f'{art_dir}/target_cols.json'):
    target_cols = json.load(open(f'{art_dir}/target_cols.json'))
if os.path.exists(f'{art_dir}/label_values.json'):
    label_values = json.load(open(f'{art_dir}/label_values.json'))
if os.path.exists(f'{art_dir}/clippings.json'):
    clippings = json.load(open(f'{art_dir}/clippings.json'))


# also check root-level json files (in case artifacts/ was not uploaded)
if os.path.exists(f'{MODEL_DATASET_DIR}/target_cols.json'):
    target_cols = json.load(open(f'{MODEL_DATASET_DIR}/target_cols.json'))
if os.path.exists(f'{MODEL_DATASET_DIR}/label_values.json'):
    label_values = json.load(open(f'{MODEL_DATASET_DIR}/label_values.json'))
if os.path.exists(f'{MODEL_DATASET_DIR}/clippings.json'):
    clippings = json.load(open(f'{MODEL_DATASET_DIR}/clippings.json'))

for c in ['question_title','question_body','answer']:
    test[c] = test[c].fillna('').map(html.unescape)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
print('targets:', len(target_cols), 'artifact_dir_exists:', os.path.exists(art_dir))


In [ ]:
def snap_to_known_labels(preds, target_cols, label_values):
    out = np.zeros_like(preds)
    for j, c in enumerate(target_cols):
        vals = np.array(label_values[c], dtype=np.float32)
        idx = np.abs(preds[:, [j]] - vals.reshape(1, -1)).argmin(axis=1)
        out[:, j] = vals[idx]
    return out

def apply_clipping(preds, cols, clippings):
    out = preds.copy()
    for j, c in enumerate(cols):
        if c in clippings:
            lo, hi = clippings[c]
            out[:, j] = np.clip(out[:, j], lo, hi)
    return out


In [ ]:
class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_len_q=256, max_len_a=256):
        self.title = df['question_title'].tolist()
        self.body = df['question_body'].tolist()
        self.answer = df['answer'].tolist()
        self.tok = tokenizer
        self.max_len_q = max_len_q
        self.max_len_a = max_len_a

    def __len__(self):
        return len(self.title)

    def __getitem__(self, idx):
        q = self.tok(self.title[idx], self.body[idx], truncation=True, max_length=self.max_len_q, padding='max_length', return_tensors='pt')
        a = self.tok(self.title[idx], self.answer[idx], truncation=True, max_length=self.max_len_a, padding='max_length', return_tensors='pt')
        return {
            'q_input_ids': q['input_ids'][0],
            'q_attention_mask': q['attention_mask'][0],
            'a_input_ids': a['input_ids'][0],
            'a_attention_mask': a['attention_mask'][0],
        }

class DualTowerRegressor(nn.Module):
    def __init__(self, model_path, n_targets=30):
        super().__init__()
        self.q_encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        self.a_encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.q_encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_targets)
        )

    def pool(self, out, mask):
        cls = out[:, 0]
        mask = mask.unsqueeze(-1)
        mean = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return torch.cat([cls, mean], dim=1)

    def forward(self, q_input_ids, q_attention_mask, a_input_ids, a_attention_mask):
        q = self.q_encoder(input_ids=q_input_ids, attention_mask=q_attention_mask).last_hidden_state
        a = self.a_encoder(input_ids=a_input_ids, attention_mask=a_attention_mask).last_hidden_state
        q_vec = self.pool(q, q_attention_mask)
        a_vec = self.pool(a, a_attention_mask)
        h = torch.cat([q_vec[:, :q_vec.shape[1]//2], a_vec[:, :a_vec.shape[1]//2]], dim=1)
        out = self.head(self.dropout(h))
        return torch.sigmoid(out)


In [ ]:
MODEL_ZOO = {
    'roberta_base': '/kaggle/input/roberta-base',
    'roberta_large': '/kaggle/input/robertalarge',
    'deberta_v3_base': '/kaggle/input/debertav3base'
}
N_FOLDS = 5

all_test = []

for model_name, model_path in MODEL_ZOO.items():
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    ds = QADataset(test, tokenizer, 256, 256)
    loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2)

    model_test = np.zeros((len(test), len(target_cols)), dtype=np.float32)

    for fold in range(N_FOLDS):
        ckpt = f'{MODEL_DATASET_DIR}/{model_name}_fold{fold}.pt'
        if not os.path.exists(ckpt):
            ckpt = f'{MODEL_DATASET_DIR}/{model_name}_fold{fold}_swa.pt'

        model = DualTowerRegressor(model_path, len(target_cols)).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device))
        model.eval()

        preds = []
        with torch.no_grad():
            for b in loader:
                p = model(
                    b['q_input_ids'].to(device),
                    b['q_attention_mask'].to(device),
                    b['a_input_ids'].to(device),
                    b['a_attention_mask'].to(device),
                ).detach().cpu().numpy()
                preds.append(p)

        preds = np.concatenate(preds)
        model_test += preds / N_FOLDS

        del model
        gc.collect(); torch.cuda.empty_cache()

    all_test.append(model_test)

ensemble_test = np.mean(np.stack(all_test, axis=0), axis=0)


In [ ]:
# final postprocess: clipping + snap to valid labels
pred = apply_clipping(ensemble_test, target_cols, clippings)
pred = snap_to_known_labels(pred, target_cols, label_values)

sub = sample.copy()
sub[target_cols] = pred
sub.to_csv('/kaggle/working/submission.csv', index=False)

# support validation
invalid_total = 0
for c in target_cols:
    allowed = set(label_values[c])
    invalid_total += int((~sub[c].isin(allowed)).sum())
print('invalid_total:', invalid_total)
sub.head()
